# Module 1 — From Business Question to Ontology

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS

---

## What this module teaches

Most ontology programs fail before a single triple is written. A small group of architects,
isolated from the consumers of the ontology, builds a model that no one outside the group
understands or trusts. The model is technically correct and practically ignored.

This module teaches a different starting point: the business question.

You will take a fuzzy, real-world question — the kind a line-of-business manager would
actually ask — and derive a small, justified ontology from it. Every class and every property
you produce will have a one-sentence rationale keyed to the question that produced it.

By the end of this module you can:

- Extract nouns and verbs from a competency question and sort them into classes, properties, and instances
- Explain the difference between a class (has independent identity) and a property (describes another thing)
- Write constraints in plain English before encoding them in OWL or SHACL
- Produce a Turtle file that is defensible in front of a Model Risk Management (MRM) reviewer
- Understand why the ATLAS starter ontology (`atlas-core.ttl`) is shaped the way it is

## What this module does NOT do

It does not start with FIBO (the Financial Industry Business Ontology). FIBO alignment
is Module 2. Starting with FIBO is a common mistake — it gives architects a vocabulary
before they have a question, and vocabularies without questions produce models no one uses.

It does not use the LLM to write the ontology. The LLM is a Socratic mirror: it asks
questions that surface decisions you need to make. You make the decisions.

## Prerequisites

- An AWS account with Amazon Bedrock model access enabled in us-east-1
- No prior ontology experience required

## Deliverables

- `ontology/atlas-core.ttl` — the 18-class starter ontology (already in the repo; you will
  read and understand it, then optionally extend it)
- `ontology/rationale.md` — one-sentence justification per class, keyed to competency questions

## Architecture class for this module

**DETERMINISTIC.** Ontology derivation from competency questions is a deterministic process:
given the same questions and the same derivation procedure, a reasonably trained ontologist
produces the same classes. The Bedrock LLM in this module is used only as a Socratic partner
that holds up a mirror — it does not write the ontology.

In [ ]:
# Workshop dependency setup — installs into THIS kernel's Python interpreter.
# Uses absolute path + cwd='/tmp' to avoid pip's os.getcwd() failure in SageMaker.
import sys, subprocess, os

req = os.path.join(os.getcwd(), 'shared', 'requirements.txt')
if not os.path.exists(req):
    # Fallback: install the key packages directly
    pkgs = ['rdflib==7.0.0', 'pyshacl==0.25.0', 'requests==2.31.0',
            'numpy==1.26.4', 'pandas==2.2.2', 'pyarrow==14.0.2',
            'faker==24.3.0', 'xgboost==2.1.4', 'scikit-learn==1.5.2']
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs,
        cwd='/tmp'
    )
else:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', req],
        cwd='/tmp'
    )
print('Dependencies ready.')


## Cell 2 — Setup

Install and import the shared ATLAS utilities. These are the same utilities used in every
subsequent module. Confirm the `rdflib` and `pyshacl` versions match what is pinned in
`notebooks/shared/requirements.txt`.

In [ ]:
# -----------------------------------------------------------------------
# ATLAS Workshop — Environment Configuration
#
# Set ATLAS_SAGEMAKER_ENV to match your SageMaker environment BEFORE
# running any other cells in this notebook.
#
#   "studio"         SageMaker Studio (JupyterLab 4, post-2023 default)
#   "studio-classic" SageMaker Studio Classic (JupyterLab 1/3, older)
#
# How to tell which you have:
#   Studio         — orange/teal Studio logo in the top bar, Launcher tab present
#   Studio Classic — dark JupyterLab interface, no Studio branding bar
#
# Both environments use the same IAM execution role and relative file paths,
# so most notebook cells are identical. This flag is used by infrastructure
# cells in later modules and makes your environment choice explicit.
# -----------------------------------------------------------------------

ATLAS_SAGEMAKER_ENV = "studio"  # <-- change to "studio-classic" if needed

_SUPPORTED_ENVS = {"studio", "studio-classic"}
assert ATLAS_SAGEMAKER_ENV in _SUPPORTED_ENVS, (
    f"ATLAS_SAGEMAKER_ENV must be 'studio' or 'studio-classic'. Got: '{ATLAS_SAGEMAKER_ENV}'"
)

_ENV_LABEL = {
    "studio":         "SageMaker Studio (JupyterLab 4)",
    "studio-classic": "SageMaker Studio Classic (JupyterLab 1/3)",
}
print(f"Environment : {_ENV_LABEL[ATLAS_SAGEMAKER_ENV]}")
print("Change ATLAS_SAGEMAKER_ENV above if this does not match your environment.")

In [ ]:
%pip install -q -r ../notebooks/shared/requirements.txt

import sys
import boto3
sys.path.insert(0, "../notebooks/shared")

import rdflib
import pyshacl
import atlas_sparql
import atlas_validators
import atlas_synthetic

print(f"rdflib   : {rdflib.__version__}")
print(f"pyshacl  : {pyshacl.__version__}")
print(f"\nATLAS shared utilities loaded.")
print(f"Synthetic data seed : {atlas_synthetic.ATLAS_SEED}")

# Print the IAM identity in use so participants can confirm the correct
# execution role is active before any AWS calls are made.
try:
    sts = boto3.Session().client("sts")
    identity = sts.get_caller_identity()
    print(f"\nAWS identity")
    print(f"  Account : {identity['Account']}")
    print(f"  Role    : {identity['Arn']}")
except Exception as exc:
    print(f"\nCould not resolve AWS identity: {exc}")
    print("Ensure the SageMaker execution role has sts:GetCallerIdentity permission.")

## What Are Competency Questions — and Why Not User Stories?

Before you read the seven starter questions below, understand what a Competency
Question (CQ) is and why it is the tool of choice for ontology engineering.

### The short version

A **Competency Question** is a natural-language question the ontology must be able
to answer once built. It is written from the consumer's perspective and acts as both
a scoping device and an acceptance test. A CQ is satisfied when you can write a
SPARQL query against the ontology that returns the right answer — this is what makes
them executable acceptance criteria, not just requirements.

### How CQs relate to other artifacts you may know

| Artifact | What It Is | Example | Scope |
|----------|-----------|---------|-------|
| **Business scenario** | An end-to-end narrative that strings together multiple user stories | "A Consumer customer deposits $500K, triggering a wealth-signal workflow that routes to an advisor" | Executive demos |
| **User story** | A value statement from a persona's perspective | "As a Wealth advisor, I want to see incoming referrals with full household context so I can prepare for outreach" | Workflow and value |
| **Competency Question** | A testable question the ontology must answer | "Which Consumer households have a member with a wealth-eligible liquidity event in the last 30 days?" | Ontology structure |
| **SPARQL query** | The implementation that proves a CQ is satisfied | `SELECT ?household WHERE { ... }` | Code |

The practical hierarchy for ATLAS is:

```
Business scenario (the demo you show a CIO)
  → User stories (advisor, RM, compliance perspectives)
    → Competency Questions (the testable layer that drives ontology structure)
      → SPARQL queries (the implementation that proves CQs are satisfied)
```

A single user story typically decomposes into multiple CQs. For example, the user
story "detect wealth-eligible events" produces these Competency Questions:
- "What constitutes a wealth-eligible event?"
- "Which Consumer customers have crossed the eligibility threshold?"
- "What is the household composition of customer X?"
- "Who is the assigned Wealth advisor for household Y?"

Each CQ forces a modeling decision — you can't answer the household question without
modeling household membership relations explicitly.

### Why CQs and not just requirements?

Traditional requirements ("the system shall...") describe system behavior.
Competency Questions are a subset of ontology requirements — specifically the ones
that test whether the model has the right concepts, relationships, and granularity.
They prevent two common failure modes:

- **Over-modeling** — concepts no one will ever query. If a class or property
  doesn't participate in answering any CQ, it's probably scope creep.
- **Under-modeling** — gaps you only discover at integration time. If you can't
  write a SPARQL query for a CQ, the ontology is incomplete.

### The four roles CQs play across this workshop

You will see these same Competency Questions reappear throughout the workshop,
each time serving a different purpose:

| Module | CQ Role | What Happens |
|--------|---------|-------------|
| 1 (here) | **Validation** | CQs prove the ontology has the right structure |
| 7 | **Grounding** | CQs become few-shot examples that constrain the LLM |
| 7 | **Accuracy** | CQs define what "correct" means for generated queries |
| 8 | **Proof of value** | CQs are answered end-to-end in the CIO demo |

Keep this in mind as you read the seven questions below. They are not throwaway
requirements — they are the single artifact that will serve as acceptance test,
vocabulary constraint, regression suite, and correctness definition across the
entire workshop.

## Cell 4 — The Seven Competency Questions

A **competency question** is a question the ontology must be able to answer. It is written
in the language the line of business actually uses, not in the language of data engineering.
The question must contain stakes — a reader who does not know the answer should care.

The workshop ships a starter set of seven competency questions for the wealth-signal use case.
Read each one aloud. Notice the nouns. Notice the verbs.

---

**CQ1.** Which Consumer-side customers have, in the last 90 days, generated an in-bank signal
that suggests wealth-management eligibility?

**CQ2.** For a given signal, which observations support it, and what is the deterministic
component of the resulting score versus the probabilistic component?

**CQ3.** Which household relationships in the bank's data make this customer a stronger or
weaker candidate, and what is the evidence for the household membership?

**CQ4.** Has this customer been previously surfaced as a wealth candidate, and if so, what
was the outcome and what has changed since?

**CQ5.** Which steps in the routing workflow require human review, and where is each review
evidenced in the graph?

**CQ6.** If a regulator asks why a particular customer was contacted by a Wealth advisor,
what is the audit trail from signal detection through advisor approval?

**CQ7.** If a customer were to ask the bank what data was used to surface them as a candidate,
can the bank answer with reference to specific in-bank observations and their dates?

---

These are not abstract database questions. They are questions a Chief Data Officer,
a model risk management reviewer, or a compliance officer would actually ask.

---

**Skill to take away:** A question is a good competency question if a domain expert
who has never seen your data model immediately understands what is at stake. The test
is to read it to a compliance officer or a CDO and watch their face — if they nod,
you have a competency question. If they ask "what do you mean by signal?", you have
a data dictionary entry dressed up as a question.

**For your own domain:** Before the next module, write one competency question for a
use case in your institution. Bring it back — we will apply the same five steps to it.

> **Proctor note:** Ask the room: "Which of these seven questions would your CDO
> actually lose sleep over?" That anchors the questions to real business stakes and
> surfaces domain-specific variations early. Watch for participants who rewrite the
> question in technical terms ("which customers have a signal record where...") —
> that is the key anti-pattern to redirect.

## Cell 5 — Step 1: Extract the Nouns

**What we are doing and why**

The first step in building any ontology is to read your competency questions and
underline every noun. Nouns are the candidates for *things* in your model — the
entities that will eventually become classes. We are not making decisions yet;
we are collecting raw material.

Read CQ1 aloud and underline every noun:

> *Which **Consumer-side customers** have, in the last **90 days**, generated an in-bank
> **signal** that suggests **wealth-management eligibility**?*

Candidates: **customer**, **90 days** (a time window), **signal**, **eligibility**.

Now CQ2:

> *For a given **signal**, which **observations** support it, and what is the
> **deterministic component** of the resulting **score** versus the **probabilistic component**?*

New candidates: **observation** (this will become Transaction and Holding),
**score**, **deterministic component**, **probabilistic component**.

Do the same for CQ3 through CQ7. The cell below captures the full noun inventory.

---

**How the "Note" column is decided**

Every noun in the inventory has a one-line note. That note answers one question:
*what is this noun's job in answering the competency question that produced it?*

The note is not a definition you look up. It is a decision you write down based on
reading the question carefully. Here is how to derive it:

1. Re-read the competency question that contains this noun.
2. Ask: "What role does this noun play in the answer?" Is it the thing being asked
   about? The evidence for the answer? The agent doing something? The time period?
3. Write that role in plain English. One sentence. No jargon.

Example:
- The noun `Customer` appears in CQ1. CQ1 asks *"Which customers..."* — Customer
  is the primary subject of the question. Note: "Primary subject of wealth-signal detection."
- The noun `90-day window` appears in CQ1. Its role is to bound the time period
  of observation. Note: "Time-bounded observation period — becomes ObservationWindow."

The note is also your first hint at whether this noun will become a class or a
property. If the note describes the noun as *having* dates, owners, and outcomes
of its own, it is probably a class. If the note describes the noun as *describing*
another noun, it is probably a property. We will make that call formally in Step 3.

**Why nouns before verbs?**

You will notice we extract nouns in Step 1 and verbs in Step 2, before sorting
anything into classes and properties in Step 3. This is intentional. Jumping to
"is this a class?" before you have listed all the nouns causes ontologists to
miss concepts — they start designing before they have finished reading. Collect
everything first, decide second.

---

> **LLM assist:** Once you have your own competency questions, you can ask Bedrock
> to do a first-pass noun extraction: *"Read this competency question and list every
> noun that could be a concept in a knowledge graph. Do not classify them yet — just
> list them."* Use the output as a starting checklist, then read the question yourself
> to catch anything the model missed. The model is fast; you are the one who knows
> whether a concept matters in your domain.

In [ ]:
# Noun inventory extracted from the seven competency questions.
# Each entry: (noun_candidate, source_questions, note)

noun_inventory = [
    ("Customer",           ["CQ1","CQ3","CQ4","CQ6","CQ7"], "Primary subject of wealth-signal detection"),
    ("90-day window",       ["CQ1","CQ4"],                    "Time-bounded observation period — becomes ObservationWindow"),
    ("Signal",              ["CQ1","CQ2","CQ5","CQ6","CQ7"], "An in-bank indicator of wealth eligibility — becomes WealthSignal"),
    ("Eligibility",         ["CQ1","CQ4"],                    "A formal determination, not just a flag — has its own identity"),
    ("Observation",         ["CQ2","CQ7"],                    "Raw evidence — resolves into Transaction and Holding"),
    ("Score",               ["CQ2"],                          "Probabilistic-explainable numeric output (XGBoost + SHAP)"),
    ("Deterministic component", ["CQ2"],                      "Part of Score — the rule-based threshold crossing"),
    ("Probabilistic component", ["CQ2"],                      "Part of Score — the ML-model output"),
    ("Household relationship",  ["CQ3"],                      "Becomes Household and HouseholdMembership"),
    ("Evidence (household)",    ["CQ3"],                      "Basis for household membership assertion"),
    ("Previous surfacing",  ["CQ4"],                          "Prior eligibility determination with outcome — becomes PreviousSurfacing"),
    ("Outcome",             ["CQ4","CQ5","CQ6"],             "Review decision (approve / decline / defer)"),
    ("Routing workflow",    ["CQ5"],                          "The Step Functions state machine — becomes WorkflowStep sequence"),
    ("Human review",        ["CQ5","CQ6"],                    "A required approval step — becomes HumanReview"),
    ("Audit trail",         ["CQ6","CQ7"],                    "The chain from signal to approval — becomes AuditRecord"),
    ("Advisor",             ["CQ5","CQ6"],                    "The human reviewer persona — becomes Advisor"),
    ("In-bank observation", ["CQ7"],                          "Specific dated fact — Transaction or Holding"),
    ("Data used",           ["CQ7"],                          "Source lineage — becomes DataSource + PROV-O attribution"),
]

print(f"{'Noun candidate':<30} {'CQs':<25} {'Note'}")
print("-" * 90)
for noun, cqs, note in noun_inventory:
    print(f"{noun:<30} {', '.join(cqs):<25} {note}")

## Cell 7 — Step 2: Extract the Verbs

**What we are doing and why**

With the noun list in hand, we now read the competency questions again — this time
looking for verbs. Verbs describe the *relationships* between things and the
*actions* that things do. In an ontology, verbs become **properties**.

There are two kinds of properties:

- **Object property** — a relationship *between two things*. Example: a Customer
  *generates* a WealthSignal. The verb "generates" connects two nouns.
- **Datatype property** — an attribute *of one thing* that is a simple value like
  a number, date, or text string. Example: a Transaction *has an amount* of USD 350,000.
  The "amount" is not a separate thing — it is a value attached to the Transaction.

---

**How the property names were derived**

For each verb we found, we chose a property name by asking two questions:

1. *What is being said about what?* — This gives us the direction: which noun is the
   "from" end and which is the "to" end.
2. *What is the most precise verb?* — We replace vague verbs ("has", "is related to")
   with specific ones that cannot be misread by a future developer or a compliance auditor.

Examples from the competency questions:

- CQ1 says a customer *"generated"* a signal. We make that directional:
  Customer **produces** WealthSignal. Name: `atlas:producesSignal`.
  The parenthetical `(Customer → WealthSignal)` shows the direction — Customer is
  the starting point (the **domain**) and WealthSignal is the ending point (the **range**).

- CQ2 says *"observations support"* a signal. The support goes from the observation
  to the signal. Name: `atlas:evidencedBy`. The signal is *evidenced by* the transaction.

---

**What Domain and Range mean**

These two terms will appear throughout the workshop. They are simple:

- **Domain** — the class at the *start* of a relationship. The thing doing or owning.
- **Range** — the class at the *end* of a relationship. The thing being pointed at.

Think of it like a sentence structure: `[domain] → verb → [range]`.

| Plain English | Domain | Property | Range |
|---------------|--------|----------|-------|
| A Customer generates a WealthSignal | Customer | producesSignal | WealthSignal |
| A WealthSignal is evidenced by a Transaction | WealthSignal | evidencedBy | Transaction |
| A RoutingDecision is reviewed by a HumanReview | RoutingDecision | reviewedBy | HumanReview |

When the domain and range are explicit, a machine can check whether you are connecting
the right kinds of things. If someone accidentally writes that a Transaction *produces*
an Advisor, the ontology can flag it as wrong — because `producesSignal` has domain
Customer, not Transaction. This is why we define domain and range precisely.

---

**The property list from our competency questions**

Each bullet below shows: the verb as written in the question, the formal property name
we chose, and the direction (domain → range).

- CQ1: "**generated** a signal" → `atlas:producesSignal` (Customer → WealthSignal)
- CQ2: "**observations support** a signal" → `atlas:evidencedBy` (WealthSignal → Transaction)
- CQ3: "**make** this customer a stronger candidate" → modelled via `HouseholdMembership.confidence` — affects Score
- CQ3: "**evidenced** by household membership" → `atlas:hasMembership` with `atlas:membershipBasis` (Customer → HouseholdMembership)
- CQ4: "**surfaced** as a candidate" → `atlas:hasPreviousSurfacing` (Customer → PreviousSurfacing)
- CQ5: "**require** human review" → `atlas:reviewedBy` (RoutingDecision → HumanReview)
- CQ5: "**evidenced** in the graph" → `atlas:hasAuditRecord` (any entity → AuditRecord)
- CQ6: "**contacted by** advisor" → `atlas:conductedBy` (HumanReview → Advisor)
- CQ6: "**triggers** routing" → `atlas:triggersRouting` (Eligibility → RoutingDecision)

**Why tense and voice matter:** "Was contacted" is passive past tense — it implies
a completed action with a specific date and a named actor. Both of those facts must
be stored somewhere. That is why `HumanReview` carries a `reviewDate` and a
`conductedBy` link to an Advisor, rather than being just a boolean flag on the
RoutingDecision. The grammar of the question tells you what data you need to keep.

---

> **LLM assist:** Once you have your own noun list, you can ask Bedrock to help
> draft property names: *"Given these two concepts — [X] and [Y] — and this
> competency question, suggest a precise verb that describes the relationship
> from X to Y. Name it in camelCase."* The model is good at this. Check that the
> direction is correct (domain → range) before accepting the suggestion — models
> occasionally reverse the direction on passive-voice verbs.

In [ ]:
# Verb inventory — verbs from the competency questions mapped to ontology properties

verb_inventory = [
    ("generated (a signal)",    "CQ1", "atlas:producesSignal",       "Customer → WealthSignal"),
    ("support (signal)",        "CQ2", "atlas:evidencedBy",           "WealthSignal → Transaction"),
    ("make stronger candidate", "CQ3", "(modelled via HouseholdMembership.confidence)", "affects Score"),
    ("evidenced by",            "CQ3", "atlas:hasMembership + atlas:membershipBasis",   "Customer → HouseholdMembership"),
    ("surfaced as candidate",   "CQ4", "atlas:hasPreviousSurfacing",   "Customer → PreviousSurfacing"),
    ("require human review",    "CQ5", "atlas:reviewedBy",             "RoutingDecision → HumanReview"),
    ("evidenced in graph",      "CQ5", "atlas:hasAuditRecord",         "any entity → AuditRecord"),
    ("contacted by",            "CQ6", "atlas:conductedBy",            "HumanReview → Advisor"),
    ("triggers routing",        "CQ6", "atlas:triggersRouting",        "Eligibility → RoutingDecision"),
    ("asked what data was used","CQ7", "atlas:sourceDataset + prov",   "entity → DataSource via PROV-O"),
    ("holds account",           "CQ1", "atlas:hasAccount",             "Customer → Account"),
    ("has holding",             "CQ1", "atlas:hasHolding",             "Account → Holding"),
    ("has transaction",         "CQ1", "atlas:hasTransaction",         "Account → Transaction"),
    ("member of household",     "CQ3", "atlas:memberOf",               "Customer → Household"),
    ("within window",           "CQ1", "atlas:withinWindow",           "WealthSignal → ObservationWindow"),
    ("processed in step",       "CQ5", "atlas:processedInStep",        "RoutingDecision → WorkflowStep"),
]

print(f"{'Verb (in question)':<30} {'CQ':<6} {'Ontology property':<45} {'Domain → Range'}")
print("-" * 110)
for verb, cq, prop, dr in verb_inventory:
    print(f"{verb:<30} {cq:<6} {prop:<45} {dr}")

## Cell 9 — Step 3: Sort Nouns into Classes, Properties, and Instances

**What we are doing and why**

We now have two lists: nouns (Step 1) and verbs (Step 2). Step 3 takes the nouns
and makes a formal decision about each one: is it a **class**, a **property**, or
a named **instance**? We use the verb list as a guide — how a noun is connected to
other nouns tells us a lot about what it is.

**Definitions — the building blocks of an ontology**

An ontology is, at its core, a vocabulary of agreed-upon concepts and the rules
about how those concepts relate to each other. Three types of things go into it:

| Term | What it is | Real-world analogy |
|------|-----------|-------------------|
| **Class** | A category of things — every member of the category is called an *instance* | "Employee" is a class; Jane Smith is an instance of Employee |
| **Property** | A relationship or attribute that connects or describes things | "works for" (connects Employee to Company); "salary" (describes Employee with a number) |
| **Instance** | A specific, named individual that belongs to a class | The specific person "Jane Smith" — not a category, a real thing |

**Who decides that these definitions are authoritative?**

The vocabulary for building ontologies — the words "class", "property", "instance" —
comes from two W3C (World Wide Web Consortium) standards that are recognised internationally:

- **OWL** (Web Ontology Language) — the standard for defining classes and their
  logical relationships. Published by W3C. Used across FSI, healthcare, government,
  and scientific publishing for interoperable knowledge models.
- **SHACL** (Shapes Constraint Language) — the standard for writing rules about what
  data in those classes must look like. Also published by W3C. In ATLAS we use SHACL
  to mechanically enforce the boundary between deterministic and probabilistic data
  (more on this in Module 6).

ATLAS uses both: OWL to define the shape of the ontology, SHACL to enforce the rules
at runtime. They are not alternatives to each other — they do different jobs.

**What is a URI?**

When you see `atlas:Customer` or `atlas:producesSignal`, the `atlas:` prefix is
shorthand for a **URI** (Uniform Resource Identifier). A URI is simply a web address
used as a unique name — not necessarily a page you can visit, but a globally unique
label that no one else can accidentally reuse.

For example, `atlas:Customer` expands to:
`https://github.com/your-org/atlas/ontology#Customer`

This long-form name guarantees that if two organisations independently build
ontologies, their concept of "Customer" and ours cannot be confused — they have
different URIs. Throughout this workshop you will see the short `atlas:` form;
the full URI is always there behind the prefix.

---

**The rule of thumb for sorting nouns:**
- If a noun has its own identity that other things will point to → **class**
- If it describes another noun and carries no independent identity → **property**
- If it is one specific named individual → **instance**

**The hard cases — worked through with the three-test framework:**

`Eligibility` — class or a boolean property of Customer?
**Class.** CQ4 asks what the *outcome* was and what has *changed since*. An Eligibility
determination has a date, a basis, a reviewer, and an outcome of its own. It needs
to exist as a thing in its own right so you can query it directly.

`Score` — a single decimal number on WealthSignal, or a class?
**Class.** CQ2 asks for the *deterministic component* vs the *probabilistic component*
separately. We need to define those terms:

- **Deterministic** means the output is fully explained by fixed rules. Given the same
  input, you always get the same output. Example: "if deposit > $250k in 90 days,
  flag as LargeDepositPattern." No ambiguity, no model.
- **Probabilistic** means a machine-learning model produced the output. Given the same
  input, the output could vary across model versions. It requires an *explanation*
  (SHAP values in ATLAS) to be defensible to a regulator.

Both components live on the same Score node. A single decimal can only hold one number;
it cannot hold the model version, the SHAP attributions, and the explainability flag
that a compliance reviewer needs to see alongside that number.

`ObservationWindow` — two date properties, or a class?
**Class.** CQ1 says "in the last 90 days" and CQ4 says "what has changed since."
If the 90-day definition changes for regulatory reasons, a class instance is one
update; two separate date columns on every signal is a bulk update across the table.

`HouseholdMembership` — a simple link (Customer → Household) or a class?
**Class when evidence matters.** CQ3 asks for *evidence* of the membership.
A simple link (`atlas:memberOf`) cannot carry evidence — it is just a pointer.
When household membership is inferred by an entity-resolution model (probabilistic),
a `HouseholdMembership` node is required to hold the basis and confidence score.
When membership comes directly from the system of record (deterministic — shared
address, same tax ID), the simple `atlas:memberOf` link is sufficient.

This is your first encounter with the deterministic-vs-probabilistic boundary —
the core architectural commitment of ATLAS. More on this in Modules 5 and 6.

---

**The three-test framework** — apply this to every noun in your own domain:

| Test | Question to ask | If it fails → it stays a property |
|------|-----------------|----------------------------------|
| Identity | Does this concept need its own unique address (URI) so other concepts can point to it? | If nothing else needs to reference it, a simple value is enough |
| History | Must multiple versions of this concept coexist for the same parent? | If only the current value matters, store it as an attribute |
| Queryability | Will you directly filter, group, sort, or count this concept in a query? | If it only ever appears as a leaf value, a literal attribute is simpler |

All three tests must pass for a noun to become a class.

**Skill to take away:** This three-test framework transfers directly to your own domain.
In Module 2 you will use it again when deciding whether your classes are equivalent
to FIBO classes or just subclasses — the same three questions apply.

> **LLM assist:** For any noun you are unsure about, ask Bedrock: *"Apply the
> three-test framework (identity, history, queryability) to the concept [X] in the
> context of this competency question: [CQ]. Should it be a class or a property?
> Explain each test."* Treat the response as a debate partner, not a final answer —
> you know your domain's compliance and audit requirements better than the model does.

> **Proctor note:** The hardest argument will be about `Score`. Data warehouse
> practitioners will want it as a column. Ask: "If a regulator asks which features
> drove this score for this customer on this date, where does that answer live if
> Score is a decimal on WealthSignal?" That closes it.
> For `ObservationWindow`, ask: "If the 90-day window changes to 60 days next quarter
> for regulatory reasons, how many rows do you update with two date columns versus one
> class instance?"

In [ ]:
# Sorted noun inventory — classes, properties, and instances

classes = [
    "Customer", "Account", "Holding", "Transaction", "Household",
    "WealthSignal", "Eligibility", "Score", "RoutingDecision",
    "HumanReview", "AuditRecord", "Advisor", "WorkflowStep",
    "WealthSignalType", "HouseholdMembership", "DataSource",
    "ObservationWindow", "PreviousSurfacing",
]

# Properties are captured in the verb inventory above and in atlas-core.ttl
# Key datatype properties derived here:
datatype_properties = [
    ("atlas:scoreValue",       "Score",             "xsd:decimal",  "Numeric value in [0,1]"),
    ("atlas:probabilistic",    "(any entity)",      "xsd:boolean",  "True if derived from probabilistic source"),
    ("atlas:confidence",       "(any entity)",      "xsd:decimal",  "Confidence score for probabilistic assertions"),
    ("atlas:explainability",   "(any entity)",      "xsd:boolean",  "True when SHAP attributions accompany the value"),
    ("atlas:modelVersion",     "(any entity)",      "xsd:string",   "Model identifier for probabilistic-explainable assertions"),
    ("atlas:selectedRoute",    "RoutingDecision",   "xsd:string",   "Must be member of atlas:RoutingRouteScheme"),
    ("atlas:reviewOutcome",    "HumanReview",       "xsd:string",   "APPROVED | DECLINED | DEFERRED"),
    ("atlas:membershipBasis",  "HouseholdMembership","xsd:string",  "Evidence description for household assertion"),
]

# Named instances (fictional persona only)
named_instances = [
    ("atlas:AlexMorgan", "atlas:Advisor", "The synthetic wealth advisor persona used in the workshop demo"),
]

print(f"Classes ({len(classes)}):")
for i, c in enumerate(classes, 1):
    print(f"  {i:2d}. atlas:{c}")

print(f"\nKey datatype properties ({len(datatype_properties)}):")
print(f"  {'Property':<30} {'Domain':<25} {'Range':<15} {'Note'}")
print("  " + "-" * 90)
for prop, domain, range_, note in datatype_properties:
    print(f"  {prop:<30} {domain:<25} {range_:<15} {note}")

print(f"\nNamed instances ({len(named_instances)}):")
for iri, cls, note in named_instances:
    print(f"  {iri}  a {cls}  # {note}")

## Cell 11 — Step 4: Write the Constraints in Plain English

**What is a constraint?**

A constraint is a rule that says something *must be true* about data in the ontology.
It is not a suggestion. If data violates a constraint, the system flags it as invalid.

Examples of constraints in plain English:
- "Every WealthSignal must have exactly one signal type." (mandatory relationship)
- "A Score value must be a number between 0 and 1." (range restriction)
- "A RoutingDecision cannot be made by an LLM acting alone — it must go through a HumanReview." (boundary rule)

Constraints are the place where the ontology stops being just a vocabulary and
starts being a *governance tool*. Without constraints, the graph is descriptive
but not enforceable — anyone can put anything in it.

---

**How constraints are enforced in ATLAS — SHACL**

In this workshop, constraints are written in **SHACL** (Shapes Constraint Language),
the W3C standard for defining and validating rules on graph data. We introduced SHACL
briefly in Step 3 alongside OWL. To be precise about which tool does what:

- **OWL** defines the *structure*: what classes exist, how they relate, what properties
  belong to what. OWL is the vocabulary.
- **SHACL** defines the *rules*: what data is valid, what is required, what is
  forbidden. SHACL is the enforcement layer.

The reason ATLAS uses SHACL specifically — not just OWL restrictions — is that SHACL
produces a **machine-readable validation report**. When a regulator asks "how do you
know the probabilistic model output never directly drove a routing decision without
human review?", the answer is: run the SHACL validator and produce the report.
OWL can express the same constraint logically, but SHACL is the tool that generates
the auditable evidence. Both are used; SHACL is the enforcement you will run and share.

SHACL shapes are written in full in Module 6. In this step, we are writing the rules
in plain English first — the plain-English form is itself a deliverable, because an
MRM reviewer reads English, not Turtle.

---

**The four questions every constraint answers**

For each class, write out the constraints by answering four questions:

1. **Identity** — What makes one instance distinct from another? (e.g., a Score is
   distinct by its WealthSignal parent, its model version, and its timestamp)
2. **Must have** — What properties must every instance have? (mandatory fields)
3. **Cannot have** — What is explicitly forbidden? (the boundary rules)
4. **Relationships** — What are the cardinality rules? ("exactly one", "at least one",
   "at most one" on each connecting property)

The cell below documents these four questions for the three classes that sit at the
deterministic-vs-probabilistic boundary — the boundary SHACL will enforce mechanically
in Module 6.

> **LLM assist:** Once you have classes from your own domain, ask Bedrock to help
> draft the plain-English constraints: *"For the class [X] in an FSI knowledge graph,
> answer these four questions: What makes one instance distinct? What must it have?
> What is it forbidden from having? What are its required relationships?"* Then review
> the output against your compliance requirements — the model will get the structure
> right but may miss institution-specific rules.

In [ ]:
# Plain-English constraints for boundary-critical classes
# These become OWL restrictions and SHACL shapes in Modules 2 and 6 respectively.

plain_english_constraints = {
    "atlas:Score": {
        "identity"    : "A Score is distinct from another Score by its WealthSignal parent, its model version, and its generation timestamp.",
        "must_have"   : ["atlas:scoreValue (xsd:decimal in [0,1])",
                         "atlas:probabilistic = true",
                         "atlas:explainability = true (SHAP attributions attached)",
                         "atlas:modelVersion (a versioned model identifier)",
                         "prov:wasGeneratedBy (the SageMaker endpoint run)"],
        "cannot_have" : ["atlas:explainability = false when used as a compliance input",
                         "A Score cannot directly drive a RoutingDecision without a HumanReview in the path"],
        "relationships": ["Exactly one parent WealthSignal (via atlas:hasScore inverse)"],
    },
    "atlas:RoutingDecision": {
        "identity"    : "A RoutingDecision is distinct by the WealthSignal it routes and the timestamp of selection.",
        "must_have"   : ["atlas:selectedRoute (one of the three SKOS route concepts)",
                         "prov:wasGeneratedBy (the Step Functions execution ARN)",
                         "atlas:processedInStep (a WorkflowStep)"],
        "cannot_have" : ["atlas:selectedRoute with a value NOT in atlas:RoutingRouteScheme",
                         "A RoutingDecision generated by an LLM free-reasoning path"],
        "relationships": ["Exactly one HumanReview downstream (before outbound action)",
                           "Exactly one Eligibility upstream (via atlas:triggersRouting inverse)"],
    },
    "atlas:HouseholdMembership": {
        "identity"    : "A HouseholdMembership is distinct by the Customer, the Household, and the date of the inference.",
        "must_have"   : ["atlas:membershipBasis (plain-English evidence description)",
                         "atlas:confidence (decimal in [0,1]) when derived probabilistically",
                         "prov:wasGeneratedBy (the Entity Resolution workflow run) when probabilistic"],
        "cannot_have" : ["atlas:confidence absent when atlas:probabilistic = true"],
        "relationships": ["Exactly one Customer", "Exactly one Household"],
    },
}

for cls, c in plain_english_constraints.items():
    print(f"\n{cls}")
    print(f"  Identity   : {c['identity']}")
    print(f"  Must have  :")
    for m in c["must_have"]:   print(f"               - {m}")
    print(f"  Cannot have:")
    for n in c["cannot_have"]: print(f"               - {n}")
    print(f"  Relationships:")
    for r in c["relationships"]: print(f"               - {r}")

## Cell 13 — Step 5: Using an LLM as a Thinking Partner

**The problem this step solves**

The five-step ontology derivation is a reasoning process, not a lookup. The hardest
part is not writing the Turtle syntax — it is making the right structural decisions:
Is this a class or a property? Does this constraint belong here or somewhere else?
What am I missing?

Those questions benefit from a thinking partner. In the past, that meant a second
ontologist in the room. In this workshop, we use Amazon Bedrock — not to make the
decisions, but to ask the questions that surface them.

**What the LLM is doing in this step — and what it is not doing**

The LLM is playing a single, narrow role here: it asks clarifying questions, the way
a good consultant would in a design review.

- It is **not** writing the ontology.
- It is **not** deciding whether something is a class or a property.
- It is **not** scoring, routing, or performing any reasoning about your data.

This distinction matters because later in the ATLAS architecture (Module 7), the same
Bedrock service plays a different role: translating natural-language questions into
SPARQL queries that run against the knowledge graph. That is a well-defined,
bounded task with a verifiable output. We keep the LLM in those defined roles
throughout — it is never the decision-maker.

In this step, the role is even simpler: **mirror**. You state a problem (is
`Eligibility` a class or a property?), and the LLM asks you three questions that
help you think through your answer. You supply the answers. The LLM does not decide.

**Why this matters beyond the workshop**

Every time you encounter a new concept in your domain and need to decide how to model
it, you can use this exact pattern. The three Socratic questions the LLM generates
will vary, but they will always probe the same axes: identity, history, and
queryability. You are learning a reusable method, not a one-time exercise.

**What to watch for in the output**

The questions Bedrock generates will be different each time you run this cell. That
is expected — the model is not producing a fixed script. What you should look for:

1. Does each question address a different aspect of the decision? (If all three ask
   essentially the same thing, re-run — the model can occasionally produce redundant questions)
2. Can you answer each question with a clear yes or no? (If a question is too vague
   to answer, it is not a good Socratic question — that is useful feedback on the model's output)
3. Does your set of answers point clearly to one conclusion?

Amazon Bedrock is pre-enabled in this workshop environment. The cell uses the
SageMaker execution role — no credentials configuration is required.

**If the cell fails**, check that:
- Your SageMaker execution role has `bedrock:InvokeModel` permission for the
  `us.anthropic.claude-sonnet-4-6` cross-region inference profile
- Your workshop account has Amazon Bedrock enabled in `us-east-1`

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError

# SageMaker execution role is used automatically — no profile or credential
# configuration required in the workshop environment.
bedrock = boto3.Session().client("bedrock-runtime", region_name="us-east-1")

# Cross-region inference profile — required for on-demand throughput in us-east-1.
_BEDROCK_MODEL = "us.anthropic.claude-sonnet-4-6"

SOCRATIC_PROMPT = """You are helping an FSI architect derive an ontology from a competency question.
Your role is Socratic: ask clarifying questions that surface the architect's design decisions.
Do NOT write ontology code. Do NOT make design decisions. Only ask questions.

The architect has identified 'Eligibility' as a noun candidate from this competency question:

  'Has this customer been previously surfaced as a wealth candidate, and if so,
   what was the outcome and what has changed since?'

The architect is deciding whether Eligibility should be:
  (a) a class (with its own instances, dates, and properties), or
  (b) a boolean property on the Customer class.

Ask exactly three clarifying questions that would help the architect make this decision.
Number each question. Keep each under 40 words."""

try:
    response = bedrock.invoke_model(
        modelId=_BEDROCK_MODEL,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 400,
            "messages": [{"role": "user", "content": SOCRATIC_PROMPT}]
        })
    )
    result = json.loads(response["body"].read())
    socratic_output = result["content"][0]["text"]

except ClientError as exc:
    code = exc.response["Error"]["Code"]
    if code == "AccessDeniedException":
        raise RuntimeError(
            "Bedrock AccessDeniedException: the SageMaker execution role does not have "
            "bedrock:InvokeModel permission for the cross-region inference profile "
            f"'{_BEDROCK_MODEL}'. Add the permission in IAM and restart the kernel."
        ) from exc
    if code == "ResourceNotFoundException":
        raise RuntimeError(
            f"Bedrock ResourceNotFoundException: inference profile '{_BEDROCK_MODEL}' "
            "was not found in us-east-1. Verify that cross-region inference is enabled "
            "for your account in the AWS Console under Amazon Bedrock > Model access."
        ) from exc
    if code == "ValidationException":
        raise RuntimeError(
            f"Bedrock ValidationException calling '{_BEDROCK_MODEL}': {exc}. "
            "Confirm the model ID matches a valid cross-region inference profile "
            "(prefix 'us.' is required for on-demand throughput in us-east-1)."
        ) from exc
    raise

print("=" * 70)
print("ATLAS Socratic Session — Bedrock as Mirror")
print("=" * 70)
print()
print("ARCHITECT: I have identified 'Eligibility' as a noun from CQ4.")
print("           I need to decide: class or boolean property?")
print()
print("BEDROCK SOCRATIC PARTNER:")
print(socratic_output)
print()
print("-" * 70)
print("Stop here. Answer the three questions above before running the next cell.")
print("-" * 70)

### Pause — answer before continuing

Write your answers to the three questions in a comment or a scratchpad before running the next cell.

The questions the model generates will differ from run to run — that is expected. What matters is the reasoning pattern, not the specific wording.

When you have your answers, run the next cell to see the reference reasoning used to derive `atlas:Eligibility` in the starter ontology.

In [ ]:
# Reference reasoning — how atlas:Eligibility was decided for the starter ontology.
# This is not LLM output. It is the derivation record for this specific class.
# Your own answers may differ; what matters is that you can justify your conclusion
# with the same three axes: identity, history, queryability.

print("REFERENCE REASONING — atlas:Eligibility")
print("=" * 70)
print()
print("Q: Does the Eligibility determination have a date, a basis, and a reviewer,")
print("   or is it just a yes/no state on Customer?")
print("A: It has a date, a basis, and a reviewer. History matters.")
print()
print("Q: If a customer is evaluated twice — declined, then approved six months later —")
print("   must both determinations be stored?")
print("A: Yes. CQ4 asks 'what has changed since' — that comparison requires both records.")
print()
print("Q: Will you query Eligibility directly — e.g., filter by outcome or date range?")
print("A: Yes — 'all Eligibility decisions in the last year with outcome DECLINED'.")
print()
print("-" * 70)
print("CONCLUSION: atlas:Eligibility is a CLASS, not a property on Customer.")
print()
print("  Three tests that force this decision:")
print("  1. Identity  — an Eligibility instance has its own date, reviewer, and outcome.")
print("  2. History   — multiple instances per Customer must coexist without overwriting.")
print("  3. Queryability — SPARQL will GROUP BY, FILTER, and ORDER BY Eligibility nodes.")
print()
print("  A boolean property fails all three tests.")

---

## Your Turn — Run the Socratic Session on Your Own Domain

The Bedrock Socratic pattern is not specific to `atlas:Eligibility` or wealth-signal
detection. The same three questions — identity, history, queryability — surface the
right decision for any noun in any FSI domain.

In the cell below, replace the two variables:

- `YOUR_NOUN` — a noun from a use case in your institution (examples: `CreditEvent`,
  `ConsentRecord`, `LimitBreach`, `TradeExecution`, `KYCAssessment`)
- `YOUR_CQ` — the competency question that produced that noun, in plain business language

Run the cell. Bedrock will ask you three clarifying questions. Write your answers as
comments in the cell or in a scratchpad. Then apply the three-test framework:
if the noun passes all three tests (identity, history, queryability), it is a class.

> **Proctor note:** Circulate while participants write their competency questions.
> The two most common errors: (1) the noun is already a column name, which means the
> participant is thinking in tables rather than questions — redirect to the business
> language test from Cell 4; (2) the competency question contains the noun they are
> trying to classify, which is circular — ask them to rewrite the question without
> naming the concept they want to model.

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError

# -----------------------------------------------------------------------
# Replace these two values with your own noun and competency question.
# -----------------------------------------------------------------------
YOUR_NOUN = "CreditEvent"   # <-- replace with your noun candidate

YOUR_CQ = (
    "Which mortgage applicants in Q3 had a credit event that our model scored "
    "as low-risk but that a human reviewer later reversed, and why?"
)  # <-- replace with your competency question
# -----------------------------------------------------------------------

_BEDROCK_MODEL = "us.anthropic.claude-sonnet-4-6"
bedrock = boto3.Session().client("bedrock-runtime", region_name="us-east-1")

your_prompt = f"""You are helping an FSI architect derive an ontology from a competency question.
Your role is Socratic: ask clarifying questions that surface the architect's design decisions.
Do NOT write ontology code. Do NOT make design decisions. Only ask questions.

The architect has identified '{YOUR_NOUN}' as a noun candidate from this competency question:

  '{YOUR_CQ}'

The architect is deciding whether '{YOUR_NOUN}' should be:
  (a) a class (with its own instances, dates, and properties), or
  (b) a property on another class.

Ask exactly three clarifying questions using the identity, history, and queryability
framework. Number each question. Keep each under 40 words."""

try:
    response = bedrock.invoke_model(
        modelId=_BEDROCK_MODEL,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 400,
            "messages": [{"role": "user", "content": your_prompt}]
        })
    )
    result = json.loads(response["body"].read())
    your_socratic_output = result["content"][0]["text"]
except ClientError as exc:
    code = exc.response["Error"]["Code"]
    if code == "AccessDeniedException":
        raise RuntimeError(
            f"Bedrock AccessDeniedException: execution role missing bedrock:InvokeModel "
            f"for '{_BEDROCK_MODEL}'."
        ) from exc
    raise

print("=" * 70)
print(f"Your Socratic Session — {YOUR_NOUN}")
print("=" * 70)
print()
print(f"COMPETENCY QUESTION: {YOUR_CQ}")
print()
print("BEDROCK SOCRATIC PARTNER:")
print(your_socratic_output)
print()
print("-" * 70)
print("Apply the three-test framework to your answers:")
print("  Identity     — Does this noun have its own URI other nodes will reference?")
print("  History      — Must multiple instances coexist for the same parent?")
print("  Queryability — Will SPARQL FILTER, GROUP BY, or ORDER BY target it directly?")
print()
print("Record your conclusion below as a comment, then continue to the noun extraction.")
print()
# Write your conclusion here:
# YOUR_NOUN is a [ CLASS / PROPERTY ] because:
#   Identity     : ...
#   History      : ...
#   Queryability : ...

## Cell 15 — Load and Inspect the Starter Ontology

The ATLAS starter ontology (`ontology/atlas-core.ttl`) was derived by applying
the five-step journey to the seven competency questions above. Load it with rdflib
and verify the class count and the competency-question traceability.

This cell is **deterministic** — it parses a file and queries it. Given the same
file, it always produces the same output.

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from rdflib.namespace import SKOS
from pathlib import Path

ATLAS = Namespace("https://github.com/your-org/atlas/ontology#")
ONTOLOGY_PATH = Path("../ontology/atlas-core.ttl")

g = Graph()
g.parse(str(ONTOLOGY_PATH), format="turtle")

# Count classes
classes = list(g.subjects(RDF.type, OWL.Class))
print(f"Classes loaded       : {len(classes)}")

# Count object properties
obj_props = list(g.subjects(RDF.type, OWL.ObjectProperty))
print(f"Object properties    : {len(obj_props)}")

# Count datatype properties
dt_props = list(g.subjects(RDF.type, OWL.DatatypeProperty))
print(f"Datatype properties  : {len(dt_props)}")

# Verify every class has rdfs:comment or skos:definition
classes_without_comment = []
for cls in classes:
    has_comment = (cls, RDFS.comment, None) in g
    has_definition = (cls, SKOS.definition, None) in g
    if not (has_comment or has_definition):
        classes_without_comment.append(str(cls))

if classes_without_comment:
    print(f"\nWARNING: {len(classes_without_comment)} class(es) missing rdfs:comment or skos:definition:")
    for c in classes_without_comment:
        print(f"  {c}")
else:
    print(f"\nAll {len(classes)} classes have rdfs:comment or skos:definition. (Rationale gate: PASS)")

# Print the class list
print("\nClass inventory:")
for i, cls in enumerate(sorted(str(c) for c in classes), 1):
    local_name = cls.split("#")[-1]
    comment_nodes = list(g.objects(g.store, None))
    print(f"  {i:2d}. atlas:{local_name}")

## Cell 17 — Load the SKOS Codelists

The `WealthSignalType` class uses a SKOS (Simple Knowledge Organization System)
concept scheme to enumerate the five v1.0 signal types. Load the codelists and
confirm all five types are present.

SKOS Reference: https://www.w3.org/TR/skos-reference/

In [ ]:
SKOS_PATH = Path("../ontology/extensions/skos-codelists.ttl")

g_skos = Graph()
g_skos.parse(str(SKOS_PATH), format="turtle")

# Merge into the main graph for unified querying
g += g_skos

# Query for WealthSignalType concepts
wst_query = """
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?concept ?label ?notation WHERE {
    ?concept skos:inScheme atlas:WealthSignalTypeScheme ;
             skos:prefLabel ?label ;
             skos:notation ?notation .
}
ORDER BY ?notation
"""

results = list(g.query(atlas_sparql.validate(wst_query)))

print(f"Wealth Signal Types in SKOS scheme: {len(results)}")
print()
print(f"  {'Label':<35} {'Notation'}")
print("  " + "-" * 60)
for row in results:
    print(f"  {str(row.label):<35} {str(row.notation)}")

assert len(results) == 5, f"Expected 5 WealthSignalTypes, got {len(results)}"
print("\nSKOS codelist loaded. 5 of 5 signal types present.")

## Cell 19 — Test the Competency Questions in SPARQL

The validation gate for Module 1 is: every competency question must be answerable
**in principle** by traversing the ontology. "In principle" means: a SPARQL query
that follows the ontology's class and property structure returns the correct shape
of result, even against a graph with only a few synthetic triples.

This cell loads a small synthetic instance graph and runs a CQ-shaped query
for each of the seven competency questions. The `atlas_sparql.validate()` wrapper
is called on every query before it touches the graph.

In [ ]:
from rdflib import Graph, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, XSD, OWL
import datetime

ATLAS = Namespace("https://github.com/your-org/atlas/ontology#")
PROV  = Namespace("http://www.w3.org/ns/prov#")
SKOS  = Namespace("http://www.w3.org/2004/02/skos/core#")

# Build a minimal instance graph that exercises all seven CQs
gi = Graph()
gi += g  # ontology + SKOS codelists already loaded

BASE = "https://github.com/your-org/atlas/instance#"

def uri(local): return URIRef(BASE + local)
def atlas_uri(local): return URIRef("https://github.com/your-org/atlas/ontology#" + local)

# --- Customer ---
cust1 = uri("customer-001")
gi.add((cust1, RDF.type,           atlas_uri("Customer")))
gi.add((cust1, atlas_uri("customerId"), Literal("C-001", datatype=XSD.string)))

# --- Account and Transaction ---
acct1 = uri("account-001")
gi.add((acct1, RDF.type,              atlas_uri("Account")))
gi.add((acct1, atlas_uri("accountType"), Literal("CHECKING", datatype=XSD.string)))
gi.add((acct1, atlas_uri("balanceUSD"),  Literal("350000.00", datatype=XSD.decimal)))
gi.add((cust1, atlas_uri("hasAccount"), acct1))

txn1 = uri("txn-001")
gi.add((txn1, RDF.type,                  atlas_uri("Transaction")))
gi.add((txn1, atlas_uri("amountUSD"),     Literal("350000.00", datatype=XSD.decimal)))
gi.add((txn1, atlas_uri("transactionType"), Literal("DEPOSIT", datatype=XSD.string)))
gi.add((txn1, atlas_uri("transactionDate"),
         Literal(str(datetime.date.today() - datetime.timedelta(days=30)), datatype=XSD.date)))
gi.add((acct1, atlas_uri("hasTransaction"), txn1))

# --- WealthSignal ---
sig1 = uri("signal-001")
gi.add((sig1, RDF.type,             atlas_uri("WealthSignal")))
gi.add((sig1, atlas_uri("signalDate"), Literal(str(datetime.date.today() - datetime.timedelta(days=25)), datatype=XSD.date)))
gi.add((sig1, atlas_uri("hasSignalType"), atlas_uri("LargeDepositPattern")))
gi.add((sig1, atlas_uri("evidencedBy"), txn1))
gi.add((cust1, atlas_uri("producesSignal"), sig1))

# --- Score (probabilistic-explainable) ---
score1 = uri("score-001")
gi.add((score1, RDF.type,                atlas_uri("Score")))
gi.add((score1, atlas_uri("scoreValue"),  Literal("0.82", datatype=XSD.decimal)))
gi.add((score1, atlas_uri("probabilistic"), Literal(True, datatype=XSD.boolean)))
gi.add((score1, atlas_uri("explainability"), Literal(True, datatype=XSD.boolean)))
gi.add((score1, atlas_uri("modelVersion"),  Literal("wealth-xgb-v1.0", datatype=XSD.string)))
gi.add((sig1,   atlas_uri("hasScore"),    score1))

# --- ObservationWindow ---
window1 = uri("window-90d")
gi.add((window1, RDF.type,                 atlas_uri("ObservationWindow")))
gi.add((window1, atlas_uri("windowStartDate"),
         Literal(str(datetime.date.today() - datetime.timedelta(days=90)), datatype=XSD.date)))
gi.add((window1, atlas_uri("windowEndDate"),
         Literal(str(datetime.date.today()), datatype=XSD.date)))
gi.add((sig1, atlas_uri("withinWindow"), window1))

# --- Household ---
hh1 = uri("household-001")
gi.add((hh1, RDF.type, atlas_uri("Household")))
gi.add((cust1, atlas_uri("memberOf"), hh1))

# --- Eligibility and RoutingDecision ---
elig1 = uri("eligibility-001")
gi.add((elig1, RDF.type, atlas_uri("Eligibility")))
gi.add((cust1, atlas_uri("hasEligibility"), elig1))

route1 = uri("routing-001")
gi.add((route1, RDF.type, atlas_uri("RoutingDecision")))
gi.add((route1, atlas_uri("selectedRoute"), Literal("ROUTE_ADVISOR_QUEUE", datatype=XSD.string)))
gi.add((elig1, atlas_uri("triggersRouting"), route1))

# --- HumanReview and Advisor ---
advisor1 = uri("advisor-alex-morgan")
gi.add((advisor1, RDF.type, atlas_uri("Advisor")))
gi.add((advisor1, RDFS.label, Literal("Alex Morgan", datatype=XSD.string)))

review1 = uri("review-001")
gi.add((review1, RDF.type,                 atlas_uri("HumanReview")))
gi.add((review1, atlas_uri("reviewOutcome"), Literal("APPROVED", datatype=XSD.string)))
gi.add((review1, atlas_uri("reviewDate"),
         Literal(datetime.datetime.now().isoformat(), datatype=XSD.dateTime)))
gi.add((route1, atlas_uri("reviewedBy"),   review1))
gi.add((review1, atlas_uri("conductedBy"), advisor1))

# --- AuditRecord ---
audit1 = uri("audit-001")
gi.add((audit1, RDF.type, atlas_uri("AuditRecord")))
gi.add((sig1,   atlas_uri("hasAuditRecord"), audit1))
gi.add((audit1, PROV.wasDerivedFrom, txn1))

# --- PreviousSurfacing ---
prev1 = uri("prev-surfacing-001")
gi.add((prev1, RDF.type, atlas_uri("PreviousSurfacing")))
gi.add((cust1, atlas_uri("hasPreviousSurfacing"), prev1))

print(f"Instance graph loaded: {len(gi)} triples (ontology + SKOS + instances)")

In [ ]:
# Run a CQ-shaped SPARQL query for each of the seven competency questions.
# Each query exercises the ontology structure relevant to that CQ.
# All queries pass through atlas_sparql.validate() before execution.

PREFIXES = atlas_sparql.build_prefixes()

cq_queries = {
    "CQ1": (
        "Which customers have generated an in-bank signal within the observation window?",
        PREFIXES + """
SELECT ?customer ?signalType ?signalDate WHERE {
    ?customer a atlas:Customer ;
              atlas:producesSignal ?sig .
    ?sig atlas:hasSignalType ?signalType ;
         atlas:signalDate ?signalDate ;
         atlas:withinWindow ?window .
}
"""
    ),
    "CQ2": (
        "For a given signal, what observations support it and what is the score decomposition?",
        PREFIXES + """
SELECT ?signal ?txn ?scoreValue ?isProbabilistic ?hasExplainability WHERE {
    ?signal a atlas:WealthSignal ;
            atlas:evidencedBy ?txn ;
            atlas:hasScore ?score .
    ?score atlas:scoreValue ?scoreValue ;
           atlas:probabilistic ?isProbabilistic ;
           atlas:explainability ?hasExplainability .
}
"""
    ),
    "CQ3": (
        "Which household relationships does this customer have?",
        PREFIXES + """
SELECT ?customer ?household WHERE {
    ?customer a atlas:Customer ;
              atlas:memberOf ?household .
}
"""
    ),
    "CQ4": (
        "Has this customer been previously surfaced as a wealth candidate?",
        PREFIXES + """
SELECT ?customer ?prevSurfacing WHERE {
    ?customer a atlas:Customer ;
              atlas:hasPreviousSurfacing ?prevSurfacing .
}
"""
    ),
    "CQ5": (
        "Which routing decisions have an associated human review?",
        PREFIXES + """
SELECT ?routing ?route ?review ?outcome WHERE {
    ?routing a atlas:RoutingDecision ;
             atlas:selectedRoute ?route ;
             atlas:reviewedBy ?review .
    ?review atlas:reviewOutcome ?outcome .
}
"""
    ),
    "CQ6": (
        "Audit trail: customer → eligibility → routing → human review → advisor",
        PREFIXES + """
SELECT ?customer ?eligibility ?routing ?review ?advisor WHERE {
    ?customer     a atlas:Customer ;
                  atlas:hasEligibility ?eligibility .
    ?eligibility  atlas:triggersRouting ?routing .
    ?routing      atlas:reviewedBy ?review .
    ?review       atlas:conductedBy ?advisor .
}
"""
    ),
    "CQ7": (
        "What specific in-bank observations were used to surface this customer?",
        PREFIXES + """
SELECT ?customer ?signal ?txn ?txnDate ?amount WHERE {
    ?customer a atlas:Customer ;
              atlas:producesSignal ?signal .
    ?signal   atlas:evidencedBy ?txn ;
              atlas:hasAuditRecord ?audit .
    ?audit    prov:wasDerivedFrom ?txn .
    ?txn      atlas:transactionDate ?txnDate ;
              atlas:amountUSD ?amount .
}
"""
    ),
}

cq_results = {}
all_pass = True

for cq_id, (description, query) in cq_queries.items():
    try:
        validated_query = atlas_sparql.validate(query)
        rows = list(gi.query(validated_query))
        cq_results[cq_id] = len(rows)
        status = "PASS" if len(rows) > 0 else "FAIL (no results)"
        if len(rows) == 0:
            all_pass = False
    except Exception as exc:
        cq_results[cq_id] = 0
        status = f"ERROR: {exc}"
        all_pass = False

    print(f"{cq_id}  [{status:^20}]  {description[:70]}")
    if rows:
        print(f"       → {len(rows)} row(s) returned")

print()
if all_pass:
    print("All 7 competency questions return non-empty results. Traversal gate: PASS.")
else:
    print("One or more competency questions returned no results. Check instance graph construction.")

## Cell 22 — Generate rationale.md

The validation gate for Module 1 requires: every class must trace to at least one
competency question. This cell generates `ontology/rationale.md` — the artifact
that makes the ontology defensible in front of a committee or a regulator.

The rationale document is not commentary. It is a deliverable.

In an MRM review, the question is not "does this model work?" It is "can you explain
every structural decision in terms the business made, before any code was written?"
`rationale.md` is that answer. If a class exists in the ontology and it does not
appear in this document, it should not exist.

**For your own domain:** This document is the template. When you extend the ontology
in a later module, every new class you add must earn a row in this table — with a
competency question citation. If you cannot write the justification sentence, the
class is premature.

> **Proctor note:** This is a good moment to check comprehension before the group
> proceeds. Ask participants to open `ontology/rationale.md` and find the row for
> `atlas:Score`. Then ask: "If your institution's equivalent of Score were just a
> decimal column in a feature store, which row in this rationale document would be
> impossible to write?" If they can answer that, the class-vs-property distinction
> from Step 3 has landed.

In [ ]:
from pathlib import Path

# Rationale map: class local name → (competency questions, one-sentence justification)
RATIONALE = {
    "Customer": (
        ["CQ1", "CQ3", "CQ4", "CQ6", "CQ7"],
        "The primary subject of wealth-signal detection; every competency question begins or ends with a Customer."
    ),
    "Account": (
        ["CQ1", "CQ2"],
        "Accounts hold the transactions and holdings that are the raw evidence for wealth signals."
    ),
    "Holding": (
        ["CQ1", "CQ2"],
        "A Customer's investment position, required to model equity-event and retirement-rollover signals."
    ),
    "Transaction": (
        ["CQ1", "CQ2", "CQ7"],
        "The atomic dated financial event that constitutes the evidence for large-deposit and business-sale signals; CQ7 requires specific dated observations."
    ),
    "Household": (
        ["CQ3"],
        "CQ3 asks about household relationships; household-aggregation signals require a class that groups Customers and supports aggregate-balance queries."
    ),
    "WealthSignal": (
        ["CQ1", "CQ2", "CQ4"],
        "The core domain concept: a typed, dated in-bank observation that indicates wealth-management eligibility."
    ),
    "Eligibility": (
        ["CQ1", "CQ4"],
        "Modelled as a class (not a boolean) because CQ4 asks for the outcome and what has changed since — both require independent identity, dates, and history."
    ),
    "Score": (
        ["CQ2"],
        "CQ2 asks for the deterministic vs probabilistic decomposition; a single decimal property cannot carry the SHAP attributions, model version, and explainability flag required."
    ),
    "RoutingDecision": (
        ["CQ5", "CQ6"],
        "CQ5 asks which steps require human review; RoutingDecision is the node that triggers HumanReview and holds the selected route from the enumerated set."
    ),
    "HumanReview": (
        ["CQ5", "CQ6"],
        "CQ5 requires evidence of human review in the graph; HumanReview carries the outcome, the date, and the task token."
    ),
    "AuditRecord": (
        ["CQ6", "CQ7"],
        "CQ6 requires a full audit trail from signal detection to advisor approval; AuditRecord is the PROV-O node that makes that trail queryable."
    ),
    "Advisor": (
        ["CQ5", "CQ6"],
        "CQ6 asks 'contacted by a Wealth advisor' — the advisor is a named participant in the audit trail and must be a queryable entity."
    ),
    "WorkflowStep": (
        ["CQ5"],
        "CQ5 asks 'which steps require human review' — WorkflowStep enumerates the bounded agent's state transitions so the graph can answer this."
    ),
    "WealthSignalType": (
        ["CQ1", "CQ2"],
        "CQ1 and CQ2 implicitly require typing signals (large-deposit vs equity-event differ in evidence and threshold); SKOS concept scheme enables closed-set SHACL enforcement."
    ),
    "HouseholdMembership": (
        ["CQ3"],
        "CQ3 asks for the evidence of household membership; reified as a class (rather than plain atlas:memberOf) when the basis and confidence score must be stored."
    ),
    "DataSource": (
        ["CQ7"],
        "CQ7 asks what data was used; DataSource is the DCAT dataset descriptor that PROV-O attribution points to, enabling source lineage queries."
    ),
    "ObservationWindow": (
        ["CQ1", "CQ4"],
        "CQ1 says 'in the last 90 days'; CQ4 asks 'what has changed since'; both require a queryable time interval with a start and end date."
    ),
    "PreviousSurfacing": (
        ["CQ4"],
        "CQ4 is entirely about whether a customer was previously surfaced and what changed; PreviousSurfacing is the node that stores the outcome and the date of the prior determination."
    ),
}

RATIONALE_PATH = Path("../ontology/rationale.md")

lines = [
    "# ATLAS Ontology — Class Rationale",
    "",
    "Every class in `atlas-core.ttl` traces to at least one competency question.",
    "This document is the Module 1 deliverable that makes the ontology defensible",
    "in front of a committee or a model risk management reviewer.",
    "",
    "| Class | Competency Questions | One-Sentence Justification |",
    "|-------|---------------------|---------------------------|",
]

for cls_name, (cqs, justification) in RATIONALE.items():
    cq_str = ", ".join(cqs)
    lines.append(f"| `atlas:{cls_name}` | {cq_str} | {justification} |")

lines += [
    "",
    "## Competency Question Coverage",
    "",
    "| CQ | Classes derived |",
    "|----|----------------|",
]

cq_to_classes = {f"CQ{i}": [] for i in range(1, 8)}
for cls_name, (cqs, _) in RATIONALE.items():
    for cq in cqs:
        cq_to_classes[cq].append(f"`atlas:{cls_name}`")

for cq, cls_list in cq_to_classes.items():
    lines.append(f"| {cq} | {', '.join(cls_list)} |")

RATIONALE_PATH.write_text("\n".join(lines) + "\n")
print(f"rationale.md written to {RATIONALE_PATH}")
print(f"Classes documented: {len(RATIONALE)}")
print()
print("Coverage check — every CQ maps to at least one class:")
for cq, cls_list in cq_to_classes.items():
    status = "PASS" if cls_list else "FAIL"
    print(f"  {cq}: {status} ({len(cls_list)} classes)")

## Cell 24 — Module 1 Validation Gate

This cell is the Module 1 validation gate. It must pass before you proceed to Module 2.

The gate checks three things:

1. **Class count** — exactly 18 OWL classes in `atlas-core.ttl`
2. **Rationale completeness** — every class has `rdfs:comment` or `skos:definition`
   (enforced by `atlas_validators.validate_ontology_completeness()`)
3. **CQ traversal** — all 7 competency questions return non-empty results
   from the instance graph

Run this cell after every change to the ontology.

In [ ]:
import sys
sys.path.insert(0, "../notebooks/shared")
from atlas_validators import validate_ontology_completeness

print("=" * 60)
print("MODULE 1 VALIDATION GATE")
print("=" * 60)

gate_pass = True

# --- Gate 1: class count ---
core_g = Graph()
core_g.parse("../ontology/atlas-core.ttl", format="turtle")
class_count = len(list(core_g.subjects(RDF.type, OWL.Class)))
expected_classes = 18

if class_count == expected_classes:
    print(f"[PASS] Gate 1 — Class count: {class_count} (expected {expected_classes})")
else:
    print(f"[FAIL] Gate 1 — Class count: {class_count} (expected {expected_classes})")
    gate_pass = False

# --- Gate 2: rationale completeness ---
result = validate_ontology_completeness(core_g, [])
if result.conforms:
    print(f"[PASS] Gate 2 — All classes have rdfs:comment or skos:definition")
else:
    print(f"[FAIL] Gate 2 — Rationale completeness:")
    for v in result.violations:
        print(f"         {v}")
    gate_pass = False

# --- Gate 3: CQ traversal ---
cq_pass_count = sum(1 for v in cq_results.values() if v > 0)
if cq_pass_count == 7:
    print(f"[PASS] Gate 3 — All 7 competency questions return non-empty results")
else:
    print(f"[FAIL] Gate 3 — {cq_pass_count}/7 competency questions return results")
    for cq_id, count in cq_results.items():
        status = "ok" if count > 0 else "EMPTY"
        print(f"         {cq_id}: {status} ({count} rows)")
    gate_pass = False

# --- Gate 4: rationale.md written ---
rationale_path = Path("../ontology/rationale.md")
if rationale_path.exists():
    line_count = len(rationale_path.read_text().splitlines())
    print(f"[PASS] Gate 4 — rationale.md present ({line_count} lines)")
else:
    print(f"[FAIL] Gate 4 — rationale.md not found at {rationale_path}")
    gate_pass = False

print()
if gate_pass:
    print("MODULE 1 VALIDATION: PASS")
    print("You may proceed to Module 2.")
else:
    print("MODULE 1 VALIDATION: FAIL")
    print("Fix the failing gate(s) above before proceeding to Module 2.")
    raise AssertionError("Module 1 validation gate failed. See output above.")

## Extending This to Your Data

The steps in this module are not specific to wealth-signal detection. They apply to any
domain where you need to build an ontology that a regulator, a CIO, or a model risk
management reviewer will need to understand.

**Replacing the competency questions with your own:**

Write three to seven questions in the language your line of business actually uses.
The test for a good competency question: if a person who knows the domain but does not know
your data model reads the question, they should immediately understand the stakes.
"What entities are in the customer table?" is not a competency question.
"Which mortgage applicants in Q3 had a credit event that our model scored as low-risk
but that a human reviewer later reversed, and why?" is a competency question.

**Most readers find their own ontology overlaps 60% with this one.**
The overlap is the FIBO-aligned core (Customer, Account, Transaction, Holding).
The divergence is where your bank-specific concepts live. That divergence is what
Module 2 teaches you to model in the extension ring.

**The three most common gotchas when replacing competency questions:**

1. **Questions that are really questions about process, not data.** "How does the approval
   workflow work?" describes a process. The ontology-relevant version is: "What is the
   evidence that step 3 of the approval workflow was completed for a given application?"
   The difference: the second question has a queryable answer.

2. **Treating every noun as a class.** A `firstName` is not a class. A `CreditEvent`
   is a class if and only if it has its own identity (a date, a type, a source) that
   you will reference from elsewhere. If it is just a boolean on the customer, it is
   a property.

3. **Skipping the verb extraction step.** The verbs are where the business logic lives.
   "A customer was contacted" implies a date, an actor, a method, and a reason — all
   of which become properties of a `ContactEvent` class. Missing the verb means
   missing the class.

---

## Your Turn — Apply the Five Steps to Your Own Competency Question

You have now seen the complete derivation: question → nouns → verbs → class/property
sorting → plain-English constraints → SPARQL traversal → rationale document.

The cell below is a structured worksheet. Replace the example competency question and
noun inventory with your own. The output will be a noun-to-class mapping that you can
carry into Module 2 as the starting point for your institution-specific ontology extension.

**What to fill in:**
1. One competency question from your domain (plain business language, stakeholder-readable)
2. The nouns you extracted from it
3. For each noun: which competency question it came from, and your one-line note

You do not need to get it right the first time. This is a working document.
The goal is to have one noun that you have run through the Socratic Bedrock cell above
and made a class-vs-property decision on.

> **Proctor note:** If the group has time, ask one participant to share their noun
> inventory on screen. Run the class-vs-property challenge live — pick the most
> ambiguous noun and work through the three-test framework together. This is the
> highest-value discussion in Module 1 for participants who bring their own domain.
> Expect 10–15 minutes. The most productive disagreements are between participants
> from different lines of business who model the same concept differently.

In [ ]:
# -----------------------------------------------------------------------
# YOUR TURN — Noun inventory worksheet
#
# Step 1: Replace YOUR_COMPETENCY_QUESTION with your own.
# Step 2: Replace the noun_inventory list entries with your own nouns.
# Step 3: Run the cell to print your noun-to-class mapping.
# Step 4: Circle back to the Socratic Bedrock cell above for any noun
#         where you are unsure about class vs property.
# -----------------------------------------------------------------------

YOUR_COMPETENCY_QUESTION = (
    # Replace this string with your own competency question.
    # Write it in the language your CDO or compliance officer would actually use.
    "Replace this with your competency question."
)

# Each entry: (noun_candidate, ["CQ_label"], "your note — what this noun might become")
# "CQ_label" is whatever label you gave your competency question (e.g., "MyCQ1").
# Add or remove rows as needed.
your_noun_inventory = [
    ("YourNoun1",   ["MyCQ1"], "Replace with your note"),
    ("YourNoun2",   ["MyCQ1"], "Replace with your note"),
    ("YourNoun3",   ["MyCQ1"], "Replace with your note"),
]

print("Competency question:")
print(f"  {YOUR_COMPETENCY_QUESTION}")
print()
print(f"{'Noun candidate':<30} {'CQs':<15} {'Note / tentative ontology mapping'}")
print("-" * 90)
for noun, cqs, note in your_noun_inventory:
    print(f"{noun:<30} {', '.join(cqs):<15} {note}")

print()
print("Next step: for each noun above, apply the three-test framework:")
print("  Identity     — Does it have its own URI?")
print("  History      — Must multiple instances coexist for the same parent?")
print("  Queryability — Will SPARQL target it directly?")
print()
print("Nouns that pass all three tests → class candidates for your Module 2 extension ring.")
print("Nouns that fail any test        → properties on an existing or new class.")

## What Changed

Module 1 added the following to the ATLAS architecture:

| Artifact | Location | Description |
|----------|----------|-------------|
| `atlas-core.ttl` | `ontology/` | 18-class starter ontology with ~30 properties, derived from 7 competency questions |
| `rationale.md` | `ontology/` | One-sentence justification per class, keyed to competency question |
| `skos-codelists.ttl` | `ontology/extensions/` | SKOS concept scheme for 5 WealthSignalTypes and 3 RoutingRoutes |
| `atlas_sparql.py` | `notebooks/shared/` | SPARQL validation wrapper — all queries pass through this before Neptune |
| `atlas_neptune.py` | `notebooks/shared/` | Neptune client with `MockNeptuneClient` for CI |
| `atlas_validators.py` | `notebooks/shared/` | SHACL and completeness validation utilities |
| `atlas_synthetic.py` | `notebooks/shared/` | Reproducible synthetic data generators (seed: 42) |

**What Module 2 builds on this:**
Module 2 takes `atlas-core.ttl` and adds FIBO IRI bindings to every class that has a
FIBO counterpart, producing `atlas-fibo-alignment.ttl`. It also documents the gaps —
classes like `WealthSignal`, `RoutingDecision`, and `HumanReview` that FIBO does not cover
and that require the extension ring (PROV-O, SKOS, DCAT, or bank-specific vocabulary).

The key decision Module 2 forces: for each class in `atlas-core.ttl`, is it
`rdfs:subClassOf` a FIBO class, or `owl:equivalentClass`, or neither?
That decision has implications for how a SPARQL reasoner treats the class and for
what SHACL shapes can assert about its instances.